# Notebook 00 — Configuración del Entorno

## Curso: Sensores Virtuales y Analizadores en Línea (Nivel Avanzado)
### Predicción en Tiempo Real de Variables de Calidad en Procesos Industriales

Este notebook verifica e instala las librerías necesarias para todo el flujo de trabajo y valida el acceso al dataset oficial de la unidad de destilación atmosférica.

**Ruta del curso (9 notebooks):**
- `00_Configuracion.ipynb`: entorno, librerías y descarga del dataset
- `01_Problema_Negocio.ipynb`: brecha de laboratorio (8 h), los 3 desafíos y *Quality Giveaway*
- `02_EDA.ipynb`: EDA industrial, colinealidad (VIF) y análisis de rezagos (*Cross-Correlation*)
- `03_Limpieza_Datos.ipynb`: duplicados, outliers físicos y limpieza sin *data leakage*
- `04_Feature_Engineering.ipynb`: gradientes térmicos, vapor específico, lags y selección en train
- `05_Modelado.ipynb`: PLS vs lineales vs árboles, Optuna, cuantiles CQR (80%) y SHAP
- `06_Evaluacion_Deriva.ipynb`: Walk-Forward, RMSE deslizante, Hotelling $T^2$/SPE y Filtro de Kalman
- `07_Despliegue.ipynb`: empaquetado `.pkl`, función de inferencia y app Streamlit HMI
- `08_Caso_Estudio_Kamyr.ipynb`: reto práctico independiente (Número Kappa en digestor Kamyr)

In [1]:
import importlib.util
import subprocess
import sys

paquetes_requeridos = [
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("sklearn", "scikit-learn"),
    ("lightgbm", "lightgbm"),
    ("optuna", "optuna"),
    ("shap", "shap"),
    ("streamlit", "streamlit"),
    ("joblib", "joblib"),
]

faltantes = [pip_name for mod_name, pip_name in paquetes_requeridos if importlib.util.find_spec(mod_name) is None]
if faltantes:
    print(f"Instalando paquetes faltantes: {faltantes}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltantes])
else:
    print("Todas las librerías requeridas ya están disponibles en el entorno.")

Todas las librerías requeridas ya están disponibles en el entorno.


In [2]:
import os
import pandas as pd
import numpy as np
import sklearn
import lightgbm as lgb
import optuna
import shap
import joblib

WORK_DIR = "/content" if os.path.isdir("/content") else "."
print(f"Directorio de trabajo activo: {os.path.abspath(WORK_DIR)}")
print("Versiones verificadas:")
print(f"  pandas       : {pd.__version__}")
print(f"  numpy        : {np.__version__}")
print(f"  scikit-learn : {sklearn.__version__}")
print(f"  lightgbm     : {lgb.__version__}")
print(f"  optuna       : {optuna.__version__}")
print(f"  shap         : {shap.__version__}")
print(f"  joblib       : {joblib.__version__}")

Directorio de trabajo activo: /Users/santiagozapata/pungo-academy/course-production/sensores-virtuales/notebooks
Versiones verificadas:
  pandas       : 2.3.3
  numpy        : 2.4.2
  scikit-learn : 1.8.0
  lightgbm     : 4.6.0
  optuna       : 4.8.0
  shap         : 0.51.0
  joblib       : 1.5.3


In [3]:
# Cargar dataset oficial desde Pungo Academy CloudFront
DATA_URL = "https://d29ikw7e1z6k4e.cloudfront.net/curso_sensor/curso_sensor_raw_data.csv"

df = pd.read_csv(DATA_URL)
print(f"Dataset descargado correctamente: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"Periodo temporal: {df['date'].iloc[0]}  ->  {df['date'].iloc[-1]} (turnos de 8 horas)")
print("\nColumnas disponibles (15 entradas de proceso + 1 fecha + 1 variable objetivo):")
for idx, col in enumerate(df.columns, 1):
    print(f"  {idx:2d}. {col} ({df[col].dtype})")

Dataset descargado correctamente: 1098 filas x 17 columnas
Periodo temporal: 2025-01-01 00:00:00  ->  2025-12-31 16:00:00 (turnos de 8 horas)

Columnas disponibles (15 entradas de proceso + 1 fecha + 1 variable objetivo):
   1. date (object)
   2. feed_flow_rate_m3h (float64)
   3. feed_temperature_C (float64)
   4. column_top_temp_C (float64)
   5. column_top_pressure_kPag (float64)
   6. column_delta_p_kPa (float64)
   7. tray_15_temp_C (float64)
   8. kerosene_draw_temp_C (float64)
   9. column_bottom_temp_C (float64)
  10. reflux_flow_m3h (float64)
  11. reflux_ratio (float64)
  12. kerosene_product_flow_m3h (float64)
  13. stripping_steam_flow_kgh (float64)
  14. condenser_outlet_temp_C (float64)
  15. ambient_temp_C (float64)
  16. crude_api_gravity (float64)
  17. flash_point_lab_C (float64)


### Checkpoint: Lo que debes ver

- Confirmación de versiones instaladas (`lightgbm`, `optuna`, `shap`, `scikit-learn`)
- Dataset con **1,098 filas** (1 año completo en turnos de 8 h + 3 retransmisiones duplicadas del SCADA) y **17 columnas**
- Rango temporal desde `2025-01-01 00:00:00` hasta `2025-12-31 16:00:00`